# Laboratorio 03: Transpilación y Comparación de Circuitos

La transpilación adapta un circuito abstracto a las restricciones físicas de un backend: **basis gates**, **conectividad** y **profundidad**. Niveles 0-3 ofrecen distintos trade-offs entre velocidad y calidad.

**Objetivo:** comparar métricas (profundidad, gates 2Q, SWAPs) entre niveles de optimización.

**Prerequisito:** Módulo 02 (Qiskit básico), Módulo 10 (Qiskit avanzado).

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.quantum_info import Operator
import numpy as np
import matplotlib.pyplot as plt
print('Entorno listo')

## 1. Circuito de referencia: QFT de 4 qubits

Usamos la QFT porque genera muchas puertas de 2 qubits y es sensible a la calidad de transpilación.

In [ ]:
from qiskit.circuit.library import QFT

# Circuito abstracto: QFT de 4 qubits
n = 4
qc = QuantumCircuit(n)
qc.compose(QFT(n, do_swaps=True), inplace=True)

print(f'Circuito abstracto QFT({n}):')
print(f'  Puertas: {qc.count_ops()}')
print(f'  Profundidad: {qc.depth()}')
print(f'  Gates 2Q: {qc.num_nonlocal_gates()}')
qc.draw('text')

## 2. Backend simulado (heavy-hex, 5 qubits)

Usamos `GenericBackendV2` para simular un backend real con conectividad limitada y basis gates específicas.

In [ ]:
# Crear backend generico con conectividad lineal (1D chain)
backend = GenericBackendV2(num_qubits=5, basis_gates=['cx','id','rz','sx','x'])
print(f'Backend: {backend.num_qubits} qubits')
print(f'Basis gates: {backend.operation_names}')
print(f'Conectividad: {list(backend.coupling_map)[:8]}...')

## 3. Transpilación en niveles 0–3

| Nivel | Estrategia | Uso típico |
|-------|-----------|------------|
| 0 | Solo mapeo de qubits (sin optimización) | Debug rápido |
| 1 | Optimización local básica | Circuitos cortos |
| 2 | Optimización media (commutation, consolidation) | **Por defecto** |
| 3 | Optimización agresiva (KAK, synthesis) | Producción |

In [ ]:
metricas = {}
for level in range(4):
    qc_t = transpile(qc, backend=backend, optimization_level=level, seed_transpiler=42)
    metricas[level] = {
        'profundidad': qc_t.depth(),
        'cx': qc_t.count_ops().get('cx', 0),
        'swaps': qc_t.count_ops().get('swap', 0),
        'total_gates': sum(qc_t.count_ops().values()),
    }
    print(f'Nivel {level}: profundidad={qc_t.depth():3d}  CX={qc_t.count_ops().get("cx",0):3d}  '
          f'gates_total={sum(qc_t.count_ops().values()):3d}')

In [ ]:
# Visualizacion comparativa
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
levels = list(metricas.keys())
colors = ['#e74c3c', '#f39c12', '#3498db', '#27ae60']

for ax, metrica, titulo in zip(
    axes,
    ['profundidad', 'cx', 'total_gates'],
    ['Profundidad del circuito', 'Puertas CX (2Q)', 'Total de puertas']):
    vals = [metricas[l][metrica] for l in levels]
    bars = ax.bar(levels, vals, color=colors, edgecolor='black', alpha=0.85)
    ax.set_xlabel('Nivel de optimizacion'); ax.set_ylabel(metrica.capitalize())
    ax.set_title(titulo); ax.set_xticks(levels)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, str(val),
                ha='center', fontweight='bold')
    ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout(); plt.savefig('transpilacion_niveles.png', dpi=100); plt.show()
print('Nivel 3 reduce CX un', round((1 - metricas[3]["cx"]/metricas[0]["cx"])*100), '% vs nivel 0')

## 4. Equivalencia unitaria: todos los niveles son correctos

In [ ]:
# Verificar que todos los circuitos son unitariamente equivalentes (mismo operador)
# (solo factible para circuitos pequenos)
qc_small = QuantumCircuit(2); qc_small.h(0); qc_small.cx(0,1); qc_small.t(1)
U_original = Operator(qc_small).data

for level in range(4):
    qc_t = transpile(qc_small, backend=backend, optimization_level=level, seed_transpiler=42)
    # Remover mediciones si las hay
    U_t = Operator(qc_t).data
    equiv = np.allclose(np.abs(U_original @ U_t.conj().T), np.eye(4), atol=1e-6)
    print(f'Nivel {level}: unitariamente equivalente = {equiv}')
print('\nConclusión: todos los niveles producen el mismo operador fisico.')